In [ ]:
import numpy as np
import os
import imageio
from tqdm import tqdm  # 用于进度条显示

# 配置参数（根据你试验成功的设定）
file_path = '/Users/apple/Downloads/cavarev_cardbreath.filtered.hq.bin'
output_dir = '/Users/apple/Desktop/images_filtered'  # 例如 'output_images'
image_shape = (960, 960)  # 替换成你成功看到图像的尺寸
dtype = np.float32  # 替换成你使用的类型，如 np.uint16、np.uint8

# 创建输出文件夹
os.makedirs(output_dir, exist_ok=True)

# 读取原始数据
data = np.fromfile(file_path, dtype=dtype)

# 总图像数
num_images = data.size // (image_shape[0] * image_shape[1])
images = data.reshape((num_images, *image_shape))

# 保存为 PNG 图像
for i in tqdm(range(num_images)):
    img = images[i]

    # 可选：归一化为 0~255 再保存（特别是 float 类型）
    img_norm = (img - img.min()) / (img.max() - img.min()) * 255
    img_uint8 = img_norm.astype(np.uint8)

    filename = os.path.join(output_dir, f'image_{i:03d}.png')
    imageio.imwrite(filename, img_uint8)


In [ ]:
# 将 MATLAB ECG-gated FDK 示例转换为 Python（结构清晰但不含完整 CT 投影几何）
import numpy as np
import struct

def read_projection_matrices(f_matrices, N):
    with open(f_matrices, 'rb') as f:
        data = np.fromfile(f, dtype=np.float32).reshape(N, 3, 4)
    return data

def read_heart_phases(f_phases, N):
    try:
        with open(f_phases, 'r') as f:
            values = np.loadtxt(f)
            if len(values) != N:
                return np.zeros(N)
            return values
    except:
        return np.zeros(N)

def get_gating_weight(phase, gprops):
    d = min([abs(phase - gprops[0]),
             abs(phase - gprops[0] - 1),
             abs(phase - gprops[0] + 1)])
    if d >= 0.5 * gprops[1]:
        return 0
    lambda_val = np.cos(d / gprops[1] * np.pi)
    return lambda_val ** 2

# 主函数框架（未执行体）
def gated_fdk(volsize, voxsize, f_projdata, f_matrices, f_phases, f_output, gating):
    N = 133
    Sx, Sy = 960, 960
    A = read_projection_matrices(f_matrices, N)
    H = read_heart_phases(f_phases, N)

    VOL = np.zeros(volsize, dtype=np.float32)

    ORIG = -voxsize * 0.5 * (np.array(volsize) - 1)
    x = np.arange(volsize[0]) * voxsize + ORIG[0]
    y = np.arange(volsize[1]) * voxsize + ORIG[1]
    z = np.arange(volsize[2]) * voxsize + ORIG[2]
    WX, WY, WZ = np.meshgrid(x, y, z, indexing='ij')
    coords = np.vstack((WX.ravel(), WY.ravel(), WZ.ravel(), np.ones(WX.size)))

    with open(f_projdata, 'rb') as f:
        for i in range(N):
            print(f"Processing image {i + 1} of {N}")
            IPP = np.fromfile(f, dtype=np.float32, count=Sx*Sy).reshape(Sy, Sx)

            A_i = A[i]
            lambda_val = get_gating_weight(H[i], gating)
            if lambda_val == 0:
                continue

            IPP *= lambda_val
            UVW = A_i @ coords
            u = UVW[0] / UVW[2] + 1
            v = UVW[1] / UVW[2] + 1

            # 简化版本：最近邻插值 + 距离加权
            u_int = np.round(u).astype(int)
            v_int = np.round(v).astype(int)
            valid = (u_int >= 0) & (u_int < Sx) & (v_int >= 0) & (v_int < Sy)
            proj_vals = np.zeros_like(u)
            proj_vals[valid] = IPP[v_int[valid], u_int[valid]] / (UVW[2][valid] ** 2)
            VOL += proj_vals.reshape(volsize)

    # 归一化并保存
    VOL = np.clip((VOL - VOL.min()) / (VOL.max() - VOL.min()) * 255, 0, 255).astype(np.uint8)

    with open(f_output, 'wb') as f:
        f.write(struct.pack('<3f', *ORIG))
        f.write(struct.pack('<3I', *volsize))
        f.write(struct.pack('<f', voxsize))
        for i in range(volsize[2]):
            slice_data = VOL[:, :, i].T
            f.write(slice_data.tobytes())

    return VOL  # 返回以便查看或后续保存为 .npy/.nii 等格式

In [ ]:
vol = gated_fdk(
    volsize=(256, 256, 256),           # 体数据尺寸
    voxsize=0.5,                       # 每个体素的尺寸（单位：mm）
    f_projdata="/Users/apple/Downloads/cavarev_cardbreath.images.hq.bin",            # 投影图像数据路径（float32）
    f_matrices="/Users/apple/Downloads/cavarev.matrices.bin",        # 3x4 投影矩阵（float32）
    f_phases="/Users/apple/Desktop/phasedata_card.txt",       # 心动相位（ASCII文本，每行一个float）
    f_output="/Users/apple/Desktop/result.vol",            # 输出保存路径（CAVAREV格式）
    gating=(0.3, 0.2)                 # gating 参数：目标心动相位 & 窗口宽度
)

In [ ]:
# 保存为 .mhd/.raw 配对文件（供 3D Slicer / MITK 使用）
def save_mhd(volume, out_prefix, spacing=1.0):
    raw_path = out_prefix + ".raw"
    mhd_path = out_prefix + ".mhd"
    volume.astype(np.uint8).tofile(raw_path)
    with open(mhd_path, "w") as f:
        f.write(f"ObjectType = Image\n")
        f.write(f"NDims = 3\n")
        f.write(f"DimSize = {volume.shape[0]} {volume.shape[1]} {volume.shape[2]}\n")
        f.write(f"ElementSpacing = {spacing} {spacing} {spacing}\n")
        f.write(f"ElementType = MET_UCHAR\n")
        f.write(f"ElementDataFile = {out_prefix.split('/')[-1]}.raw\n")

# 用法
save_mhd(vol, "volume_out")
